# VIX Futures & Options Signal Research

This notebook demonstrates VIX-based trading strategies using data from IBKR.

## 1. Setup

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import asyncio
from datetime import datetime, timedelta

# Add project root to path
project_root = Path('/Users/zelin/Desktop/PA Investment/Invest_strategy')
sys.path.insert(0, str(project_root))

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Setup complete")

✅ Setup complete


## 2. Load VIX Data from IBKR

In [2]:
from ib_insync import IB, Index, Stock
import nest_asyncio
nest_asyncio.apply()

async def load_vix_from_ibkr(duration: str = "2 Y") -> pd.DataFrame:
    """Load VIX spot and related data from IBKR."""
    ib = IB()
    
    try:
        await ib.connectAsync('127.0.0.1', 7496, clientId=1)
    except Exception as e:
        print(f"ERROR: Could not connect to IBKR: {e}")
        return pd.DataFrame()
    
    print("Connected to IBKR, fetching VIX data...")
    
    data = {}
    
    # 1. VIX Spot Index
    print("Fetching VIX Index (spot)...")
    try:
        vix = Index('VIX', 'CBOE', 'USD')
        bars = await ib.reqHistoricalDataAsync(vix, '', duration, '1 day', 'TRADES', True)
        if bars:
            data['vix_spot'] = pd.Series([b.close for b in bars], 
                                         index=pd.DatetimeIndex([b.date for b in bars]))
            print(f"   VIX spot: {len(bars)} bars")
    except Exception as e:
        print(f"   Error: {e}")
    
    # 2. VXX ETF
    print("Fetching VXX ETF...")
    try:
        vxx = Stock('VXX', 'SMART', 'USD')
        bars = await ib.reqHistoricalDataAsync(vxx, '', duration, '1 day', 'TRADES', True)
        if bars:
            data['vxx_etf'] = pd.Series([b.close for b in bars],
                                          index=pd.DatetimeIndex([b.date for b in bars]))
            print(f"   VXX: {len(bars)} bars")
    except Exception as e:
        print(f"   Error: {e}")
    
    # 3. UVXY
    print("Fetching UVXY ETF...")
    try:
        uvxy = Stock('UVXY', 'SMART', 'USD')
        bars = await ib.reqHistoricalDataAsync(uvxy, '', duration, '1 day', 'TRADES', True)
        if bars:
            data['uvxy_etf'] = pd.Series([b.close for b in bars],
                                          index=pd.DatetimeIndex([b.date for b in bars]))
            print(f"   UVXY: {len(bars)} bars")
    except Exception as e:
        print(f"   Error: {e}")
    
    ib.disconnect()
    
    if data:
        df = pd.DataFrame(data).ffill().dropna()
        print(f"Total: {len(df)} rows")
        return df
    
    print("No data loaded!")
    return pd.DataFrame()

In [3]:
# Load data directly (nest_asyncio handles Jupyter event loop)
vix_data = asyncio.run(load_vix_from_ibkr("2 Y"))

# Fallback: Generate sample data if IBKR not available
if vix_data.empty:
    print("⚠️ IBKR not connected - using sample data for testing")
    dates = pd.date_range('2022-01-01', '2024-12-31', freq='B')
    np.random.seed(42)
    # Simulate VIX-like behavior
    vix_spot = 20 + np.cumsum(np.random.randn(len(dates)) * 0.5) + 10 * np.sin(np.arange(len(dates)) * 2 * np.pi / 60)
    vix_spot = np.clip(vix_spot, 10, 80)
    vxx_etf = 30 + np.cumsum(np.random.randn(len(dates)) * 0.3)
    uvxy_etf = 15 + np.cumsum(np.random.randn(len(dates)) * 0.2)
    
    vix_data = pd.DataFrame({
        'vix_spot': vix_spot,
        'vxx_etf': vxx_etf,
        'uvxy_etf': uvxy_etf
    }, index=dates)

print("Data shape:", vix_data.shape)
vix_data.tail()

Connected to IBKR, fetching VIX data...
Fetching VIX Index (spot)...
   VIX spot: 501 bars
Fetching VXX ETF...
   VXX: 501 bars
Fetching UVXY ETF...
   UVXY: 501 bars
Total: 501 rows
Data shape: (501, 3)


,vix_spot,vxx_etf,uvxy_etf
2026-02-23,21.01,29.44,40.98
2026-02-24,19.55,28.49,38.75
2026-02-25,17.93,27.38,36.72
2026-02-26,18.63,27.72,37.44
2026-02-27,19.86,28.90,39.65


## 3. Build Features

In [4]:
if not vix_data.empty:
    # Returns
    vix_data['vix_return'] = vix_data['vix_spot'].pct_change()
    
    # Moving averages
    for window in [5, 10, 20, 60]:
        vix_data[f'ma_{window}'] = vix_data['vix_spot'].rolling(window).mean()
        vix_data[f'std_{window}'] = vix_data['vix_spot'].rolling(window).std()
    
    # Mean reversion z-score
    vix_data['zscore_20'] = (vix_data['vix_spot'] - vix_data['ma_20']) / vix_data['std_20']
    vix_data['zscore_60'] = (vix_data['vix_spot'] - vix_data['ma_60']) / vix_data['std_60']
    
    # Momentum
    vix_data['momentum_5d'] = vix_data['vix_spot'].pct_change(5)
    vix_data['momentum_20d'] = vix_data['vix_spot'].pct_change(20)
    
    # Volatility regime
    vix_data['vol_regime'] = np.where(vix_data['vix_spot'] > vix_data['ma_60'], 'HIGH', 'LOW')
    
    print("Features created:")
    print(vix_data.columns.tolist())
else:
    print("No data loaded - features skipped")

Features created:
['vix_spot', 'vxx_etf', 'uvxy_etf', 'vix_return', 'ma_5', 'std_5', 'ma_10', 'std_10', 'ma_20', 'std_20', 'ma_60', 'std_60', 'zscore_20', 'zscore_60', 'momentum_5d', 'momentum_20d', 'vol_regime']


## 4. Define Trading Signals

In [5]:
def create_signals(df):
    """Create trading signals based on VIX dynamics."""
    signals = pd.DataFrame(index=df.index)
    
    # 1. Mean Reversion: Short when VIX is high (z-score > 2)
    signals['mean_reversion'] = np.where(df['zscore_20'] > 2, -1, 
                                          np.where(df['zscore_20'] < -2, 1, 0))
    
    # 2. Trend Following: Go long when VIX is rising
    signals['trend_following'] = np.where(df['momentum_5d'] > 0, 1, -1)
    
    # 3. Regime Filter: Only trade in low VIX regime
    signals['regime_filter'] = np.where(df['vol_regime'] == 'LOW', 1, 0)
    
    # 4. Combined signal
    signals['combined'] = signals['mean_reversion'] * signals['regime_filter']
    
    return signals

if not vix_data.empty:
    signals = create_signals(vix_data)
    print("Signals created:")
    print(signals.describe())
else:
    print("No data - signals not created")

Signals created:
       mean_reversion  trend_following  regime_filter    combined
count      501.000000       501.000000     501.000000  501.000000
mean        -0.059880        -0.037924       0.610778    0.005988
std          0.283562         1.000279       0.488061    0.134030
min         -1.000000        -1.000000       0.000000   -1.000000
25%          0.000000        -1.000000       0.000000    0.000000
50%          0.000000        -1.000000       1.000000    0.000000
75%          0.000000         1.000000       1.000000    0.000000
max          1.000000         1.000000       1.000000    1.000000


In [6]:
# Patch BacktestEngine to add Broker observer (run BEFORE backtest)
# Equity curve is now built in the engine from position_history (unified with position data)
from backend.backtest_engine import BacktestEngine
import backtrader as bt

# Store original only on first run (avoids RecursionError when cell re-runs) (avoids RecursionError when cell re-runs)
if not hasattr(BacktestEngine, '_cb_original_init_cerebro'):
    BacktestEngine._cb_original_init_cerebro = BacktestEngine._init_cerebro
_original_init = BacktestEngine._cb_original_init_cerebro

def _patched_init_cerebro(self):
    _original_init(self)
    self.cerebro.addobserver(bt.observers.Broker)

BacktestEngine._init_cerebro = _patched_init_cerebro
print("✅ BacktestEngine patched!")

✅ BacktestEngine patched!


In [7]:
# Prepare OHLCV data for RegimeFilter backtest
# bt_data is used by the RegimeFilter strategy in the next cell
if 'vix_data' in locals() and not vix_data.empty:
    bt_data = pd.DataFrame({
        'open': vix_data['vix_spot'],
        'high': vix_data['vix_spot'] * 1.02,
        'low': vix_data['vix_spot'] * 0.98,
        'close': vix_data['vix_spot'],
        'volume': 1000000
    }, index=vix_data.index)
    print(f"Prepared bt_data: {len(bt_data)} bars for RegimeFilter backtest")
else:
    print("No vix_data. Load data first.")
# =============================================================================
# Regime Filter Strategy (Close-to-Close, No Look-Ahead)
# =============================================================================
# Buy when VIX < 60-day SMA (LOW regime), close when VIX > 60-day SMA (HIGH regime).
# Signal: close[0], sma[0] (current bar only - available at end of bar).
# Execution: Order.Close - fills at next bar's close (true close-to-close).
# Run the patch cell and bt_data cell above before this cell.

import backtrader as bt
from backend.backtest_engine import BacktestEngine, IBKRDataFeed

class RegimeFilterStrategy(bt.Strategy):
    """
    Buy when VIX is below its 60-day moving average.
    Sell when VIX crosses above the moving average.
    Close-to-close execution: no look-ahead bias.
    """
    
    params = (('period', 60),)  # SMA period
    
    def __init__(self):
        self.sma = bt.indicators.SimpleMovingAverage(
            self.data.close, 
            period=self.params.period
        )
        
    def next(self):
        pos = self.getposition(self.data).size
        # close[0], sma[0] = current bar only (no future data)
        close, sma = float(self.data.close[0]), float(self.sma[0])
        
        # VIX below SMA = LOW regime = BUY
        # VIX above SMA = HIGH regime = SELL/CLOSE
        if close < sma:
            # LOW regime - enter long if flat
            if pos == 0:
                self.buy(exectype=bt.Order.Close)  # fill at next bar's close
        else:
            # HIGH regime - close if long
            if pos > 0:
                self.close(exectype=bt.Order.Close)  # fill at next bar's close


# Run backtest
engine = BacktestEngine(cash=100000, commission=0.001)
engine.add_data(IBKRDataFeed(dataname=bt_data.copy()), name='VIX')
engine.add_strategy(RegimeFilterStrategy)
result = engine.run_backtest()

print("=== Regime Filter Strategy (Custom) ===")
print(f"Initial: ${result['initial_cash']:,.0f}")
print(f"Final:   ${result['final_value']:,.0f}")
print(f"Return:  {result['total_return']*100:.2f}%")

if result.get('equity_curve') is not None:
    print(f"Equity: {len(result['equity_curve'])} points")

Prepared bt_data: 501 bars for RegimeFilter backtest
=== Regime Filter Strategy (Custom) ===
Initial: $100,000
Final:   $757,968
Return:  657.97%
Equity: 442 points


## 7. Enhanced Backtest Visualization with Trade Markers & Events

This cell provides enhanced visualization features:
- Price chart with buy/sell markers
- Position time series
- Win-rate and trade statistics
- FOMC event annotations (vertical lines)
- Drawdown analysis

In [8]:
# Enhanced plotting function with trade markers and event annotations
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def metrics_summary_table(result: dict) -> pd.DataFrame:
    """Create a nicely formatted metrics table from backtest result."""
    m = result
    sharpe = m.get('sharpe_ratio') or m.get('sharperatio')
    rows = [
        ('Initial Cash', f"${m.get('initial_cash', 0):,.0f}"),
        ('Final Value', f"${m.get('final_value', 0):,.0f}"),
        ('Total Return', f"{m.get('total_return', 0)*100:.2f}%"),
        ('Sharpe Ratio', f'{sharpe:.3f}' if sharpe is not None else '—'),
        ('Max Drawdown', f"{m.get('max_drawdown', 0)*100:.2f}%"),
        ('Win Rate', f"{m.get('win_rate', 0)*100:.1f}%"),
        ('Total Trades', str(int(m.get('total_trades', 0)))),
        ('Won / Lost', f"{int(m.get('won_trades', 0))} / {int(m.get('lost_trades', 0))}"),
        ('Profit Factor', f"{m.get('profit_factor', 0):.2f}"),
        ('Volatility (Ann.)', f"{m.get('annualized_volatility', 0)*100:.2f}%" if m.get('annualized_volatility') else '—'),
        ('Sortino Ratio', f"{m.get('sortino_ratio', 0):.3f}" if m.get('sortino_ratio') else '—'),
        ('Calmar Ratio', f"{m.get('calmar_ratio', 0):.2f}" if m.get('calmar_ratio') else '—'),
    ]
    return pd.DataFrame(rows, columns=['Metric', 'Value']).set_index('Metric')


def plot_backtest_with_trades(prices, equity_curve, trades_df=None, position_series=None,
                              events_df=None, benchmark_returns=None, price_col=None, metrics=None):
    """
    Enhanced backtest visualization with trade markers, cumulative returns, and metrics table.

    Args:
        prices: Price data (DataFrame with 'close' or 'vix_spot' column)
        equity_curve: Portfolio value series
        trades_df: Trade log DataFrame
        position_series: Position size over time (1=long, 0=flat, -1=short)
        events_df: Events to annotate (DataFrame with 'date' and 'label' columns)
        benchmark_returns: Benchmark returns for comparison
        price_col: Price column name (auto-detects 'close', 'vix_spot', or first numeric col)
        metrics: Backtest result dict for metrics table (e.g. result from engine.run_backtest())
    """
    if price_col is None:
        for c in ['close', 'vix_spot', 'vxx_etf', 'uvxy_etf']:
            if c in prices.columns:
                price_col = c
                break
        if price_col is None:
            price_col = prices.select_dtypes(include=[np.number]).columns[0]
    price_series = prices[price_col]
    # Normalize equity_curve to Series (handle DataFrame with portfolio_value column)
    eq_series = equity_curve['portfolio_value'] if isinstance(equity_curve, pd.DataFrame) and 'portfolio_value' in equity_curve.columns else equity_curve

    # Stack time series vertically (full width), shared x-axis for synchronized zoom/selection
    fig = make_subplots(
        rows=6, cols=1,
        subplot_titles=(
            'Price with Trade Markers',
            'Position (% of Asset)',
            'Equity Curve',
            'Cumulative Return (%)',
            'Drawdown (%)',
            'Daily Return (%)'
        ),
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.18, 0.14, 0.18, 0.14, 0.14, 0.14]
    )

    # 1. Price chart with trade markers
    fig.add_trace(
        go.Scatter(
            x=prices.index,
            y=price_series,
            mode='lines',
            name='Price',
            line=dict(color='#2196F3', width=1.5),
            hovertemplate='%{x|%Y-%m-%d}<br>Price: %{y:.2f}<extra></extra>'
        ),
        row=1, col=1
    )

    # Add buy/sell markers from trades
    if trades_df is not None and not trades_df.empty:
        col = 'trade_type' if 'trade_type' in trades_df.columns else trades_df.columns[0]
        buys = trades_df[trades_df[col].astype(str).str.upper().str.contains('BUY', na=False)]
        sells = trades_df[trades_df[col].astype(str).str.upper().str.contains('SELL', na=False)]

        if len(buys) > 0:
            x_buy = pd.to_datetime(buys['date'] if 'date' in buys.columns else buys.index)
            y_buy = price_series.reindex(x_buy).ffill().bfill().values  # Snap markers to price curve
            valid = ~np.isnan(y_buy)
            x_buy, y_buy = x_buy[valid], y_buy[valid]
            if len(x_buy) > 0:
                fig.add_trace(
                    go.Scatter(x=x_buy, y=y_buy, mode='markers', name='Buy',
                               marker=dict(symbol='triangle-up', size=8, color='#81c784', line=dict(width=1, color='#66bb6a')),
                               hovertemplate='%{x|%Y-%m-%d}<br>BUY @ %{y:.2f}<extra></extra>'),
                    row=1, col=1
                )
        if len(sells) > 0:
            x_sell = pd.to_datetime(sells['date'] if 'date' in sells.columns else sells.index)
            y_sell = price_series.reindex(x_sell).ffill().bfill().values  # Snap markers to price curve
            valid = ~np.isnan(y_sell)
            x_sell, y_sell = x_sell[valid], y_sell[valid]
            if len(x_sell) > 0:
                fig.add_trace(
                    go.Scatter(x=x_sell, y=y_sell, mode='markers', name='Sell',
                               marker=dict(symbol='triangle-down', size=8, color='#e57373', line=dict(width=1, color='#ef5350')),
                               hovertemplate='%{x|%Y-%m-%d}<br>SELL @ %{y:.2f}<extra></extra>'),
                    row=1, col=1
                )

    # Add event annotations (FOMC)
    if events_df is not None and not events_df.empty:
        for _, event in events_df.iterrows():
            fig.add_vline(
                x=event['date'],
                line_dash="dash",
                line_color="red",
                annotation_text=event.get('label', 'Event'),
                row=1, col=1
            )

    # 2. Position over time (% of asset: 100=long, 0=flat, -100=short)
    if position_series is not None and not position_series.empty:
        pos_raw = position_series['position'] if isinstance(position_series, pd.DataFrame) and 'position' in position_series.columns else position_series
        pos_y = pos_raw * 100  # Convert to percentage
        fig.add_trace(
            go.Scatter(
                x=position_series.index,
                y=pos_y,
                mode='lines',
                name='Position %',
                fill='tozeroy',
                line=dict(color='#9C27B0', width=1.5),
                hovertemplate='%{x|%Y-%m-%d}<br>Position: %{y:.0f}%<extra></extra>'
            ),
            row=2, col=1
        )

    # 3. Equity curve
    fig.add_trace(
        go.Scatter(
            x=eq_series.index,
            y=eq_series,
            mode='lines',
            name='Strategy',
            line=dict(color='#4CAF50', width=2),
            hovertemplate='%{x|%Y-%m-%d}<br>Equity: $%{y:,.2f}<extra></extra>'
        ),
        row=3, col=1
    )

    # Add benchmark if provided
    if benchmark_returns is not None:
        benchmark_equity = (1 + benchmark_returns).cumprod() * eq_series.iloc[0]
        fig.add_trace(
            go.Scatter(
                x=benchmark_equity.index,
                y=benchmark_equity,
                mode='lines',
                name='Benchmark',
                line=dict(color='#FFC107', width=1.5, dash='dash'),
                hovertemplate='%{x|%Y-%m-%d}<br>Benchmark: $%{y:,.2f}<extra></extra>'
            ),
            row=3, col=1
        )

    # 4. Cumulative return
    cum_return = (eq_series / eq_series.iloc[0] - 1) * 100
    fig.add_trace(
        go.Scatter(
            x=cum_return.index,
            y=cum_return,
            mode='lines',
            name='Cumulative Return %',
            fill='tozeroy',
            line=dict(color='#00BCD4', width=2),
            hovertemplate='%{x|%Y-%m-%d}<br>Cum Return: %{y:.2f}%<extra></extra>'
        ),
        row=4, col=1
    )

    # 5. Drawdown
    running_max = eq_series.cummax()
    drawdown = (eq_series - running_max) / running_max * 100
    fig.add_trace(
        go.Scatter(
            x=drawdown.index,
            y=drawdown,
            mode='lines',
            name='Drawdown %',
            fill='tozeroy',
            line=dict(color='#F44336', width=1.5),
            hovertemplate='%{x|%Y-%m-%d}<br>Drawdown: %{y:.2f}%<extra></extra>'
        ),
        row=5, col=1
    )

    # 6. Daily return (%)
    daily_ret = eq_series.pct_change() * 100
    fig.add_trace(
        go.Scatter(
            x=daily_ret.index,
            y=daily_ret,
            mode='lines',
            name='Daily Return %',
            line=dict(color='#FF9800', width=1),
            hovertemplate='%{x|%Y-%m-%d}<br>Daily Return: %{y:.2f}%<extra></extra>'
        ),
        row=6, col=1
    )

    # Update layout: full width, range slider for time selection, unified hover for all curve values
    fig.update_layout(
        height=1000,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
        title_text="Enhanced Backtest Analysis",
        title_x=0.5,
        hovermode='x unified',  # Shows values for all subplots at selected time
        margin=dict(l=60, r=40, t=80, b=80)
    )
    # Y-axis labels and units
    fig.update_yaxes(title_text='Position (%)', row=2, col=1, tickformat='.0f', ticksuffix='%')
    fig.update_yaxes(title_text='Cumulative Return (%)', row=4, col=1, tickformat='.1f', ticksuffix='%')
    fig.update_yaxes(title_text='Drawdown (%)', row=5, col=1, tickformat='.1f', ticksuffix='%')
    fig.update_yaxes(title_text='Daily Return (%)', row=6, col=1, tickformat='.2f', ticksuffix='%')
    # Range slider on bottom x-axis for time selection
    fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.05), row=6, col=1)
    return fig


# Load FOMC dates for annotations
try:
    from backend.cb_meeting_schedule import get_fomc_dates
    fomc_dates = get_fomc_dates()
    print(f"Loaded {len(fomc_dates)} FOMC meeting dates")
except ImportError:
    fomc_dates = []
    print("FOMC schedule not available")


# Example: If you have run a backtest, visualize with:
fig = plot_backtest_with_trades(
    prices=vix_data,
    equity_curve=result['equity_curve']['portfolio_value'],
    trades_df=result['trade_log'],
    position_series=result['position_series'],
    events_df=pd.DataFrame(fomc_dates, columns=['date']),
    metrics=result  # Pass full result for metrics table
)
fig.show()

# Display metrics table (standalone, for easy copy/paste)
display(metrics_summary_table(result).style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1565C0'), ('color', 'white'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('padding', '8px')]}
]))
print("✅ Enhanced plotting functions loaded")

FOMC schedule not available


,Value
Metric,
Initial Cash,"$100,000"
Final Value,"$757,968"
Total Return,657.97%
Sharpe Ratio,1.684
Max Drawdown,38.26%
Win Rate,76.0%
Total Trades,25
Won / Lost,19 / 6
Profit Factor,4.04


✅ Enhanced plotting functions loaded


## 10. Live Trading with Same Engine

Use the same strategy logic from backtesting for live IBKR execution.

> **Note**: Backtrader's native IBStore (`bt.stores.IBStore`) is not available in version 1.9.78.123. The `LiveTradingEngine` uses our custom `IBKRClient` (via `ib_insync`) for live data and execution.

In [9]:
# Live Trading (optional) - use RegimeFilterStrategy with LiveTradingEngine
# Uncomment and adapt for live execution when ready
# from backend.backtest_engine import LiveTradingEngine
# import asyncio
# async def run_live():
#     engine = LiveTradingEngine(cash=100000)
#     await engine.connect_ibkr()
#     engine.add_live_data('VIX', sec_type='IND', exchange='CBOE')  # or VXX
#     engine.add_strategy(RegimeFilterStrategy)
#     await engine.run_live_async()
# asyncio.run(run_live())
